# 🚀 Notebook do Professor (Demo) — Aula 12: Router chains e o conceito de grafo de estado

**Disciplina:** Prompt Engineering and Artificial Intelligence  
**Instituição:** FIAP — Ciência da Computação · 2026  
**Professor:** Jorge Luiz Gomes  
**Aula 12/14 — Módulo 4: LangGraph e Encerramento**  
**⏱️ 1h40min**  
**🔀 Router Chain · StateGraph mental · LangGraph motivação**  
**🔁 Andaime 50%**  

---

## Como usar este notebook

- Cada célula corresponde a um slide de código da aula (a ordem é a da apresentação).
- Rode ao vivo enquanto explica o slide correspondente.
- A última seção traz o lab do aluno com o gabarito das lacunas.

---

# 🔬 Código da aula — slide a slide

### Slide 07 — Classificador de intenção — o coração do Router

In [ ]:
!pip install langchain-ollama langchain-core langchain-classic -q

from langchain_ollama import ChatOllama
from google.colab import userdata
import os

# Definir a API key via variável de ambiente (Colab Secrets)
os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from pydantic import BaseModel
from typing import Literal

llm = ChatOllama(model="gpt-oss:120b", temperature=0)

# Schema Pydantic para a saída do classificador
class Rota(BaseModel):
    destino: Literal["rag", "calculadora", "conversa"]
    # Literal garante que o LLM retorne APENAS uma dessas 3 strings
    # Se retornar algo diferente → ValidationError → fallback para "conversa"

prompt_classificador = ChatPromptTemplate.from_messages([
    ("system", """Você é um roteador de intenções. Classifique o input do usuário em exatamente um destino:
- rag: perguntas sobre documentos do domínio, manuais, contratos, regulamentos
- calculadora: cálculos matemáticos, conversões numéricas, porcentagens
- conversa: saudações, perguntas gerais, agradecimentos, qualquer outra coisa

Retorne apenas o JSON com o campo 'destino'."""),
    ("human", "{input}"),
])

# Chain do classificador — with_structured_output garante Pydantic válido
chain_classificador = prompt_classificador | llm.with_structured_output(Rota)

# Testar o classificador isoladamente
for inp in ["Qual a cláusula 5 do contrato?", "Quanto é 450 × 0.88?", "Oi tudo bem?"]:
    rota = chain_classificador.invoke({"input": inp})
    print(f"'{inp[:30]}...' → {rota.destino}")
# 'Qual a cláusula 5 do contrato?...' → rag
# 'Quanto é 450 × 0.88?...'          → calculadora
# 'Oi tudo bem?...'                  → conversa

### Slide 08 — Handlers — uma chain por rota

In [ ]:
from langchain_core.runnables import RunnableLambda

# Handler 1 — RAG (reutiliza o retriever e prompt do CKP02)
prompt_rag = ChatPromptTemplate.from_template(
    "Use o contexto abaixo para responder a pergunta. Se não souber, diga que não sabe.\n"
    "{contexto}\n\nPergunta: {input}"
)
handler_rag = (
    {"contexto": retriever | RunnableLambda(lambda docs: "\n\n".join(d.page_content for d in docs)),
     "input": lambda x: x["input"]}
    | prompt_rag | llm | StrOutputParser()
)

# Handler 2 — calculadora (sem LLM para a matemática)
def _calcular(dados: dict) -> str:
    # Primeiro extrai a expressão Python via LLM, depois calcula
    expressao = (ChatPromptTemplate.from_template(
        "Extraia apenas a expressão Python do cálculo pedido. Retorne somente a expressão, sem texto: {input}"
    ) | llm | StrOutputParser()).invoke(dados)
    try:
        resultado = eval(expressao.strip(), {"__builtins__": {}}, {})
        return f"Resultado: {resultado} (expressão: {expressao.strip()})"
    except:
        return "Não foi possível calcular. Tente escrever a expressão de outra forma."
handler_calculadora = RunnableLambda(_calcular)

# Handler 3 — conversa genérica
handler_conversa = (
    ChatPromptTemplate.from_messages([
        ("system", "Você é um assistente simpático do domínio do grupo. Responda de forma amigável e concisa."),
        ("human", "{input}"),
    ]) | llm | StrOutputParser()
)

# Mapa de handlers por destino
HANDLERS = {"rag": handler_rag, "calculadora": handler_calculadora, "conversa": handler_conversa}

### Slide 09 — Montando o Router — classificar → despachar

In [ ]:
from langchain_core.runnables import RunnableLambda

def rotear(dados: dict) -> str:
    """Classifica a intenção e despacha para o handler correto."""
    inp  = dados["input"]
    rota = chain_classificador.invoke({"input": inp})

    print(f"[ROUTER] '{inp[:40]}' → {rota.destino}")

    # Despachar para o handler correspondente
    handler = HANDLERS.get(rota.destino, HANDLERS["conversa"])  # fallback: conversa
    return handler.invoke({"input": inp})

# Router chain final — RunnableLambda torna a função uma Runnable LCEL
router_chain = RunnableLambda(rotear)

# Testar o sistema completo
testes = [
    "Qual o prazo de garantia no contrato?",   # → rag
    "Quanto é 1.250 com 15% de desconto?",     # → calculadora
    "Você pode me ajudar com algo hoje?",      # → conversa
    "Qual a multa por rescisão antecipada?",    # → rag
]
for t in testes:
    print(f"\n{'='*50}\nInput: {t}\nOutput: {router_chain.invoke({'input': t})}")

### Slide 17 — with_structured_output() — structured output revisitado

In [ ]:
# LLM pode retornar qualquer texto
"rag"          # ✅ ok
"RAG"          # ❌ KeyError no dict
"use o rag"    # ❌ KeyError no dict
"Vou usar rag" # ❌ quebra em produção

### Slide 17 — with_structured_output() — structured output revisitado

In [ ]:
# Literal garante exatamente 1 de N
class Rota(BaseModel):
    destino: Literal["rag","calc","chat"]
# O LLM é forçado (via JSON schema)
# a retornar APENAS um dos 3 valores
# "RAG" → ValidationError → tratável

### Slide 20 — Python novo desta aula

In [ ]:
# 1. Literal — tipo que aceita apenas valores específicos
from typing import Literal

def fn(cor: Literal["vermelho", "azul", "verde"]): ...
# fn("amarelo") → erro de type checking (mas não RuntimeError em Python puro)
# com Pydantic: Literal em BaseModel → ValidationError em runtime

# 2. with_structured_output(PydanticModel) — reforçar Literal em runtime
from pydantic import BaseModel

class Rota(BaseModel):
    destino: Literal["a", "b"]

chain = prompt | llm.with_structured_output(Rota)
result = chain.invoke({"input": "..."})
result.destino  # → sempre "a" ou "b" — nunca outro valor

# 3. dict.get(key, default) — fallback seguro
d = {"a": 1, "b": 2}
d.get("c", "default")  # → "default" (sem KeyError)
d["c"]                   # → KeyError (quebra em produção)

# 4. RunnableLambda — transformar qualquer função em Runnable LCEL
from langchain_core.runnables import RunnableLambda

def minha_fn(dados: dict) -> str: return dados["x"].upper()
runnable = RunnableLambda(minha_fn)
runnable.invoke({"x": "hello"})  # → "HELLO"
# RunnableLambda é o "adaptador" que coloca qualquer função no pipe LCEL

# 5. Lambda em dicts de chain — LCEL fan-out
chain = {
    "contexto": retriever | RunnableLambda(lambda docs: "\n".join(d.page_content for d in docs)),
    "input": lambda x: x["input"],
} | prompt | llm  # dict com lambdas = fan-out LCEL (executa em paralelo)

---

# 💻 Lab do aluno — versão com lacunas

## 📋 Roteiro do Lab

**Lab — Aula 12 · 2º Semestre**  
### Router Chain com 3 rotas para o domínio do grupo ★★★

*Grupo 3–4 · 30 minutos · Google Colab*

1. Complete as 4 lacunas — Literal com destinos reais do domínio, prompt do classificador descrevendo cada rota com exemplos específicos do domínio, persona do handler_conversa, função rotear completa.
2. Teste os 3 cenários — RAG (pergunta sobre documentos do grupo), calculadora (um cálculo relevante para o domínio) e conversa (saudação ou pergunta fora do escopo).
3. Analise o roteamento — troque uma palavra-chave na pergunta que vai para RAG para ver se o classificador ainda acerta. Ex: "documentação" → "manual" → "arquivo" — qual para de rotear corretamente?

> **🎯 Gabarito das lacunas**
>
> L1: Literal["rag", "calculadora", "conversa"] (ou os 3 destinos do domínio)
>
> L2: System prompt descrevendo cada destino com exemplos do domínio real
>
> L3: system="Você é um assistente de [DOMÍNIO]. Responda de forma amigável..."
>
> L4: dados["input"] ; rota.destino ; rotear

In [ ]:
!pip install langchain langchain-ollama langchain-chroma pydantic -q

from langchain_ollama import ChatOllama, OllamaEmbeddings
from langchain_chroma import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableLambda
from pydantic import BaseModel
from typing import Literal
import os
from google.colab import userdata

os.environ["OLLAMA_HOST"]    = "https://ollama.com"
os.environ["OLLAMA_API_KEY"] = userdata.get("OLLAMA_API_KEY")
llm        = ChatOllama(model="gpt-oss:120b", temperature=0)
embeddings = OllamaEmbeddings(model="nomic-embed-text")
retriever  = Chroma(persist_directory="/content/ckp02",
                    embedding_function=embeddings).as_retriever(search_kwargs={"k":3})

# 👉 LACUNA 1: Rota com Literal — adicionar os destinos do domínio do grupo
class Rota(BaseModel):
    destino: Literal[___, ___, ___]  # ex: "rag", "calculadora", "conversa"

# 👉 LACUNA 2: prompt do classificador — descrever cada destino do domínio
prompt_clf = ChatPromptTemplate.from_messages([
    ("system", """___"""),  # descrever cada rota com exemplos do domínio
    ("human", "{input}"),
])
chain_clf = prompt_clf | llm.with_structured_output(Rota)

# Handler RAG (pronto)
handler_rag = ({"contexto": retriever | RunnableLambda(lambda d: "\n".join(x.page_content for x in d)),
                "input": lambda x: x["input"]}
               | ChatPromptTemplate.from_template("Contexto:\n{contexto}\n\nPergunta: {input}")
               | llm | StrOutputParser())

# 👉 LACUNA 3: implementar handler_conversa (chain simples com system prompt amigável)
handler_conversa = (
    ChatPromptTemplate.from_messages([
        ("system", ___),  # persona do assistente do domínio do grupo
        ("human", "{input}"),
    ]) | llm | StrOutputParser()
)

HANDLERS = {"rag": handler_rag, "calculadora": handler_calculadora, "conversa": handler_conversa}

# 👉 LACUNA 4: implementar a função rotear(dados) e montar router_chain
def rotear(dados: dict) -> str:
    rota = chain_clf.invoke({"input": dados[___]})
    print(f"[ROUTER] {rota.destino}")
    return HANDLERS.get(___, HANDLERS["conversa"]).invoke(dados)

router_chain = RunnableLambda(___)

## 📚 Referências da aula

- Docs LangChain — RunnableLambda e routing patterns. Como construir chains com branching usando RunnableLambda e with_structured_output. python.langchain.com/docs/how_to/routing
- Docs LangGraph — Conceitos de StateGraph, nodes e edges. A leitura recomendada antes da Aula 13. langchain-ai.github.io/langgraph/concepts/low_level
- Blog Anthropic Engineering — "Building Effective Agents" (2025). Seção sobre orchestrators e subagents — a motivação arquitetural para grafos de estado. anthropic.com/engineering/building-effective-agents
- Livro Russell, S.; Norvig, P. — Inteligência Artificial. 3ª ed. Pearson, 2016. Cap. 3 — Resolução de problemas como busca: a teoria por trás de grafos de estado em IA, base conceitual do LangGraph.
- Livro Polzer, D. — RAG with Python Cookbook. O'Reilly, 2026. O custo de latência de um router baseado em LLM (500ms-2s) e a alternativa de classificador leve sobre embeddings — a fundamentação por trás do classificador de intenção desta aula.
- Livro Gullí, A. — Agentic Design Patterns. O'Reilly, 2025. Cap. 3 — Routing: as três formas de implementar roteamento (regras, classificador de ML, LLM) e o trade-off de custo/latência por trás do Router Chain desta aula.

---

**Próxima Aula — Aula 13** — LangGraph — StateGraph, conditional edges e HITL
  
O diagrama desta aula vira código. Estado TypedDict, add_node(), add_conditional_edges(), MemorySaver, interrupt_before.

---

*Copyright © 2026 Prof. Jorge Luiz Gomes · FIAP · Todos os direitos reservados.*